# proxytool_REDUX_4_REPRO

Slim, reproducible benchmark runbook for REDUX_4 metadata similarity.

- **Full exploratory notebook:** `proxytool_REDUX_4.ipynb` (unchanged)
- **Definitions:** extracted to `proxytool_redux/_extracted/redux4_core.py` via `scripts/extract_redux4_core.py`
- **Primary metrics:** Test 2 (functional similar) + Test 3 (dissimilar) + `30_Pairs.json` vertical cohort
- **Excluded from headline tables:** mirror identity (query = same repo) — not meaningful for accuracy claims

## Run order
1. Setup (token + flags)
2. Load core definitions (~2–5 min import/compile)
3. `run_all_benchmarks()` — single pass, shared normalizer, CSV export
4. Review per-domain summary

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

# --- Flags ---
RUN_MIRROR_APPENDIX = False  # strict 4-candidate mirror retrieval (slow; appendix only)
CLEAR_CACHE_FIRST = False    # True for cold reproducibility; False for faster iteration
MAX_COMMITS = 150
EXPORT_DIR = Path("results_benchmark")

# GitHub token: env var preferred
github_token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
if not github_token:
    github_token = None  # set here if needed: github_token = "ghp_..."
    print("Warning: no GITHUB_TOKEN / GH_TOKEN — rate limits will be tight.")
else:
    print("GitHub token loaded from environment.")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from proxytool_redux.bootstrap import load_redux4_core

t0 = time.perf_counter()
g = load_redux4_core({"github_token": github_token} if github_token else None)
if github_token:
    g["github_token"] = github_token
g["pd"] = pd
print(f"Loaded redux4_core in {time.perf_counter() - t0:.1f}s")
print(
    "Key symbols:",
    "build_argument_table" in g,
    "build_custom_30_table" in g,
    "run_known_pair_benchmark" in g,
)

In [ ]:
from proxytool_redux.benchmark import run_all_benchmarks

t0 = time.perf_counter()
results = run_all_benchmarks(
    g,
    export_dir=EXPORT_DIR,
    pairs_path="30_Pairs.json",
    max_commits=MAX_COMMITS,
    clear_cache_first=CLEAR_CACHE_FIRST,
    run_mirror_appendix=RUN_MIRROR_APPENDIX,
    make_mirror_plots=False,
    github_token=github_token,
)
elapsed = time.perf_counter() - t0
print(f"\nBenchmark finished in {elapsed / 60:.1f} min")

In [ ]:
print("=== Test 2 / Test 3 (no mirror identity) ===")
display(results["table_test2"])
display(results["table_test3"])

print("\n=== 30-pair vertical cohort ===")
display(
    results["table_30"][
        ["ID", "Domain", "Test", "Metadata", "Code centric", "Dynamic", "Cross language"]
    ]
)

print("\n=== Per-domain summary ===")
display(results["domain_summary"])

print("\n=== Discrimination (similar = Test2 + 30-pair; dissimilar = Test3) ===")
display(results["diagnostics"])